# Day 05

In [52]:
from itertools import islice

# Read and Parse Data

In [53]:
with open('input.txt') as f:
    data = f.read().split('\n')

In [54]:
def parse_data(data):
    # Divide by empty line
    sections = []
    current_section = []
    for line in data:
        if line.strip() == '':
            if current_section:
                sections.append(current_section)
                current_section = []
        else:
            current_section.append(line)
    if current_section:
        sections.append(current_section)
    return sections

data_parsed = parse_data(data)

## Data Preprocessing

In [55]:
section = data_parsed[1]
section[1:]

['3547471595 1239929038 174680800',
 '3052451552 758183681 481745357',
 '0 1427884524 1775655006',
 '2844087171 549819300 208364381',
 '3767989253 4004864866 5194940',
 '3534196909 1414609838 13274686',
 '1775655006 114264781 435554519',
 '4148908402 4010059806 146058894',
 '2729822390 0 114264781',
 '3773184193 4156118700 138848596',
 '2211209525 3203539530 518612865',
 '3912032789 3767989253 236875613']

In [56]:
def process_data(data):
    data_dict = {}
    for idx, section in enumerate(data):
        if idx == 0:
            seeds = list(map(int, section[0][7:].split(' ')))

        else:
            data_dict[section[0][:-5]] = section[1:]

    return seeds, data_dict

# Part 1

In [57]:
len(data_parsed)

8

In [58]:
# The results will be a list of numbers
# The order is: 
# [seed, soil, fertilizer, water, light, temperature, humidity, location]

seeds, data_dict = process_data(data_parsed)
results = [seeds]
while True:
    idx = len(results) - 1
    to_convert = results[-1]
    name, data = next(islice(data_dict.items(), idx, None))
    print(f"Converting: {name}")
    destination_list = []
    for item in to_convert:
        print(f"    Item: {item}")
        added = False
        for info in data:
            destination, origin, length = tuple(map(int, info.split(' ')))
            if item in range(origin, origin+length):
                print(f"        Found in range: {info}")
                print(f"        {origin<=item}, {item<=origin+length-1}")
                destination_list.append(destination + (item - origin))
                added=True
                break
        if not added:
            print(f"        Not found in any range. Adding {item} instead")
            destination_list.append(item)

    print(f"Completed section")
    results.append(destination_list)

    if len(results) == 8:
        break

Converting: seed-to-soil
    Item: 5844012
        Found in range: 2729822390 0 114264781
        True, True
    Item: 110899473
        Found in range: 2729822390 0 114264781
        True, True
    Item: 1132285750
        Found in range: 3052451552 758183681 481745357
        True, True
    Item: 58870036
        Found in range: 2729822390 0 114264781
        True, True
    Item: 986162929
        Found in range: 3052451552 758183681 481745357
        True, True
    Item: 109080640
        Found in range: 2729822390 0 114264781
        True, True
    Item: 3089574276
        Found in range: 0 1427884524 1775655006
        True, True
    Item: 100113624
        Found in range: 2729822390 0 114264781
        True, True
    Item: 2693179996
        Found in range: 0 1427884524 1775655006
        True, True
    Item: 275745330
        Found in range: 1775655006 114264781 435554519
        True, True
    Item: 2090752257
        Found in range: 0 1427884524 1775655006
        True, True
 

In [59]:
min(results[-1])

825516882

# Part 2

## First idea:

- *Description:* Start from the lowest interval in location and try to build up to a seeds number.
- *Problem:* The 'functions' are invertible but I don't necessarily have the full domain as a codomain of the previous step.

Not following this approach

## Second Idea

- *Description:* Transforming the whole interval into the next category. Possible using only the start and end (or length).
- *Problems:* Theoretically none. In implementation: I wasn't modular enough and ended up with a long code difficult to debug.

## Third Idea

- *Description:* Create a class defining an Interval variable and the methods to transform that interval according to some rules.
- *Problem:* Slower implementation.

In [60]:
class IntervalMap:
    def __init__(self, start, length) -> None:
        """
        Initialize an IntervalMap representing a discrete, inclusive integer interval.

        Args:
            start (int): Inclusive starting index of the interval.
            length (int): Number of consecutive integers in the interval (must be >= 1).

        Attributes:
            start (int): Inclusive start index (same as the provided start).
            length (int): Number of elements in the interval.
            stop (int): Inclusive end index, computed as start + length - 1.

        Notes:
            The interval represents the integer range [start, stop].
        """
        self.start = start
        self.length = length 
        self.stop = start + length - 1

    def offset_interval(self, offset : int) -> 'IntervalMap':
        """
        Return a new IntervalMap shifted by a given offset.

        Args:
            offset (int): Signed amount to shift the interval. Positive shifts move the interval
                to higher indices (right); negative shifts move it to lower indices (left).

        Returns:
            IntervalMap: A new IntervalMap with start = self.start + offset and the same length
                as the original. The original interval is not modified.

        Notes:
            The returned interval's stop is computed as (start + length - 1).
        """
        offseted_start = self.start + offset
        return IntervalMap(offseted_start, self.length)
    

    def intersection(self, other_interval: 'IntervalMap') -> 'IntervalMap | None':
        """
        Calculate the intersection of this interval with another interval.

        Args:
            other_interval (IntervalMap): Another IntervalMap instance to intersect with.

        Returns:
            IntervalMap | None: A new IntervalMap representing the intersection of the two intervals,
            or None if there is no intersection.
        """
        new_start = max(self.start, other_interval.start)
        new_stop = min(self.stop, other_interval.stop)

        if new_start <= new_stop:
            new_length = new_stop - new_start + 1
            return IntervalMap(new_start, new_length)
        else:
            return None
        

    def difference(self, other_interval: 'IntervalMap') -> 'list[IntervalMap] |IntervalMap | None':
        """
        Calculate the difference between this interval and another interval.

        Args:
            other_interval (IntervalMap): Another IntervalMap instance to subtract from this interval.
        Returns:
            IntervalMap | None: A new IntervalMap representing the difference of the two intervals,
            or None if there is no difference.
        """
        # Other interval includes self
        if other_interval.start > self.stop or other_interval.stop < self.start:
            return IntervalMap(self.start, self.length)

        # Other interval and self are disjoint
        if other_interval.start <= self.start and other_interval.stop >= self.stop:
            return None

        # Other interval is inside self
        if other_interval.start > self.start and other_interval.stop < self.stop:
            left_length = other_interval.start - self.start
            right_length = self.stop - other_interval.stop
            return [IntervalMap(self.start, left_length), IntervalMap(other_interval.stop + 1, right_length)]

        # Left intersection
        if other_interval.start <= self.start:
            new_start = other_interval.stop + 1
            new_length = self.stop - other_interval.stop
            return IntervalMap(new_start, new_length)

        # Right intersection
        if other_interval.stop >= self.stop:
            new_length = other_interval.start - self.start
            return IntervalMap(self.start, new_length)
        

    def apply_rule(self, rule: str) -> tuple['IntervalMap | None', 'list[IntervalMap] | IntervalMap | None']:
        """
        Parse a rule and compute how this interval would be transformed.

        Args:
            rule (str): A string with three integers separated by spaces:
            "<destination> <source> <size>". This represents moving the
            source interval [source, source + size - 1] so that it starts
            at index `destination`. The applied offset is (destination - source).

        Returns:
            tuple:
            offsetted_intersec (IntervalMap | None): The portion of self that
                intersects the rule's source interval, shifted by the computed
                offset. None if there is no intersection.
            remainder (IntervalMap | list[IntervalMap] | None): The part(s) of
                self that are not affected by the rule. Returns None if the
                rule fully covers self, a single IntervalMap if the remainder
                is contiguous, or a list of two IntervalMap objects if the
                source interval splits self into two pieces.

        Notes:
            - The method does not modify self; it returns new IntervalMap objects.
            - The rule string must contain three integers; invalid input will raise
              ValueError during parsing.
        """
        destination, source, size = tuple(map(int, rule.split(' ')))
        source_interval = IntervalMap(source, size)
        offset = destination - source

        intersection = self.intersection(source_interval)
        offsetted_intersec = None
        if intersection:
            offsetted_intersec = intersection.offset_interval(offset)

        remainder = self.difference(source_interval)

        return offsetted_intersec, remainder

    
    def apply_multiple_rules(self, rules, verbose: bool = False):
        def fmt(item):
            if item is None:
                return "None"
            if isinstance(item, list):
                return "[" + ", ".join(fmt(x) for x in item) + "]"
            return f"Interval(start={item.start}, length={item.length}, stop={item.stop})"

        intervals_to_process = [self]
        transformed_intervals = []
        iteration = 0

        while intervals_to_process:
            iteration += 1
            current_interval = intervals_to_process.pop()
            if verbose:
                print(f"\nIteration {iteration}: processing {fmt(current_interval)}")
            transformed = False

            for rule in rules:
                if verbose:
                    print(f"  Trying rule '{rule}' on {fmt(current_interval)}")
                offsetted_intersec, remainder = current_interval.apply_rule(rule)

                if verbose:
                    print(f"    -> intersection (shifted): {fmt(offsetted_intersec)}")
                    print(f"    -> remainder: {fmt(remainder)}")

                if offsetted_intersec:
                    transformed_intervals.append(offsetted_intersec)
                    if verbose:
                        print(f"    Added transformed interval: {fmt(offsetted_intersec)}")
                    transformed = True

                if remainder and offsetted_intersec:
                    if isinstance(remainder, list):
                        if verbose:
                            print(f"    Adding remainder pieces back to process queue: {fmt(remainder)}")
                        intervals_to_process.extend(remainder)
                    else:
                        if verbose:
                            print(f"    Adding remainder back to process queue: {fmt(remainder)}")
                        intervals_to_process.append(remainder)

                if transformed:
                    if verbose:
                        print(f"    Interval {fmt(current_interval)} was transformed by rule '{rule}', stop trying other rules.")
                    break

            if not transformed:
                transformed_intervals.append(current_interval)
                if verbose:
                    print(f"  No rule transformed {fmt(current_interval)}. Keeping as-is.")

        if verbose:
            print("\nFinal transformed intervals:")
            for t in transformed_intervals:
                print(f"  {fmt(t)}")

        return transformed_intervals
    

    def multi_intervals_multi_rules(self, intervals, rules):
        """
        Apply multiple rules to multiple intervals.

        This helper iterates over a collection of IntervalMap instances and applies the
        same ordered list of textual rules to each interval. The actual transformation
        work is delegated to each interval's apply_multiple_rules method.

        Args:
            intervals (Iterable[IntervalMap]): An iterable of IntervalMap objects to process.
            rules (Sequence[str]): A sequence (list/tuple/...) of rule strings, each in the
            form "<destination> <source> <size>" as accepted by apply_rule.

        Returns:
            None: This method populates and returns the transformed_intervals list defined
            after this docstring (the caller code continues to perform the processing).
        """
       
        transformed_intervals = []
        for interval in intervals:
            transformed = interval.apply_multiple_rules(rules)
            transformed_intervals.extend(transformed)
        return transformed_intervals
    
    def get_min_of_intervals(self, iter_intervals):
        """
        Get the minimum start value from an iterable of IntervalMap instances.

        Args:
            iter_intervals (Iterable[IntervalMap]): An iterable of IntervalMap objects.
        Returns:
            int: The minimum start value among the provided intervals.
        """
        return min(iv.start for iv in iter_intervals)

## Testing Class

### Offset, Intersection and Difference

In [61]:
# Assumes IntervalMap is already defined in the notebook.

def iv_equal(a, b):
    if a is None and b is None:
        return True
    if a is None or b is None:
        return False
    return (a.start, a.length, a.stop) == (b.start, b.length, b.stop)

def list_iv_equal(a_list, b_list):
    if a_list is None and b_list is None:
        return True
    if a_list is None or b_list is None:
        return False
    if isinstance(a_list, list) and isinstance(b_list, list):
        if len(a_list) != len(b_list):
            return False
        return all(iv_equal(x, y) for x, y in zip(a_list, b_list))
    return iv_equal(a_list, b_list)

# Offset tests
orig = IntervalMap(5, 4)   # [5,8]
off_pos = orig.offset_interval(3)
off_neg = orig.offset_interval(-2)
assert iv_equal(off_pos, IntervalMap(8, 4)), "Positive offset failed"
assert iv_equal(off_neg, IntervalMap(3, 4)), "Negative offset failed"
# original unchanged
assert iv_equal(orig, IntervalMap(5,4)), "offset_interval mutated original"

# Intersection symmetry tests
a = IntervalMap(10, 10)  # [10,19]
b = IntervalMap(15, 10)  # [15,24]
c = IntervalMap(0, 5)    # [0,4]
d = IntervalMap(5, 5)    # [5,9]
e = IntervalMap(5, 1)    # [5,5]
f = IntervalMap(5, 1)    # [5,5] identical

# overlapping
ia = a.intersection(b)
ib = b.intersection(a)
assert iv_equal(ia, IntervalMap(15, 5)), "intersection a/b incorrect"
assert iv_equal(ia, ib), "intersection not symmetric for overlapping intervals"

# disjoint (touching at boundary is disjoint here)
ic = c.intersection(d)
assert ic is None, "intersection of touching/disjoint intervals should be None"

# identical single point
id1 = e.intersection(f)
id2 = f.intersection(e)
assert iv_equal(id1, IntervalMap(5,1)), "intersection of identical single-point failed"
assert iv_equal(id1, id2), "intersection not symmetric for identical intervals"

# Difference tests
# 1) other inside self -> two pieces
self_iv = IntervalMap(10, 10)   # [10,19]
other_inside = IntervalMap(12, 3)  # [12,14]
diff = self_iv.difference(other_inside)
expected = [IntervalMap(10, 2), IntervalMap(15, 5)]  # [10,11] and [15,19]
assert list_iv_equal(diff, expected), f"difference (split) failed: got {diff}"

# 2) disjoint -> returns the same interval
self_iv2 = IntervalMap(10,4)   # [10,13]
other_disjoint = IntervalMap(0,5)  # [0,4]
diff2 = self_iv2.difference(other_disjoint)
assert iv_equal(diff2, IntervalMap(10,4)), "difference with disjoint interval should return original"

# 3) other covers self -> None
self_iv3 = IntervalMap(10,5)  # [10,14]
other_cover = IntervalMap(5, 20)  # covers [5,24]
diff3 = self_iv3.difference(other_cover)
assert diff3 is None, "difference when other covers self should be None"

# 4) left intersection (other overlaps left part of self)
self_iv4 = IntervalMap(10,10)   # [10,19]
other_left = IntervalMap(5,8)   # [5,12] overlaps [10,12]
diff4 = self_iv4.difference(other_left)
# remainder should be [13,19] => start=13 length=7
assert iv_equal(diff4, IntervalMap(13,7)), f"left intersection difference failed: got {diff4}"

# 5) right intersection (other overlaps right part of self)
self_iv5 = IntervalMap(10,10)   # [10,19]
other_right = IntervalMap(15,10)  # [15,24] overlaps right [15,19]
diff5 = self_iv5.difference(other_right)
# remainder should be [10,14] => start=10 length=5
assert iv_equal(diff5, IntervalMap(10,5)), f"right intersection difference failed: got {diff5}"

print("All IntervalMap tests passed")

All IntervalMap tests passed


### Rule based methods

In [62]:
# Tests for apply_rule and apply_multiple_rules (assumes IntervalMap, iv_equal, list_iv_equal are defined)

print("Test 1")
# 1) Rule fully covers the interval
self1 = IntervalMap(10, 5)  # [10,14]
rule_full = "20 10 5"       # maps [10,14] -> [20,24]
off, rem = self1.apply_rule(rule_full)
assert iv_equal(off, IntervalMap(20,5)), "apply_rule full-cover: offsetted incorrect"
assert rem is None, "apply_rule full-cover: remainder should be None"

print("Test 2")
# 2) Disjoint rule
rule_disjoint = "0 0 5"     # source [0,4] doesn't touch [10,14]
off2, rem2 = self1.apply_rule(rule_disjoint)
assert off2 is None, "apply_rule disjoint: offsetted should be None"
assert iv_equal(rem2, self1), "apply_rule disjoint: remainder should equal original"

print("Test 3")
# 3) Left overlap
self2 = IntervalMap(10, 10)  # [10,19]
rule_left = "0 5 8"         # source [5,12] overlaps left part [10,12]
off3, rem3 = self2.apply_rule(rule_left)
assert iv_equal(off3, IntervalMap(5,3)), f"apply_rule left-overlap: offsetted wrong, got {off3}"
# remainder for that case should be [13,19] => start=13 length=7
assert iv_equal(rem3, IntervalMap(13,7)), f"apply_rule left-overlap: remainder wrong, got {rem3}"

print("Test 4")
# 4) Right overlap
rule_right = "0 15 10"      # source [15,24] overlaps right [15,19]
off4, rem4 = self2.apply_rule(rule_right)
assert iv_equal(off4, IntervalMap(0,5)), "apply_rule right-overlap: offsetted wrong"
assert iv_equal(rem4, IntervalMap(10,5)), "apply_rule right-overlap: remainder wrong"

print("Test 5")
# 5) Other inside self (splits into two pieces)
rule_inside = "100 12 3"    # source [12,14] inside self2
off5, rem5 = self2.apply_rule(rule_inside)
assert iv_equal(off5, IntervalMap(100,3)), f"apply_rule inside: offsetted wrong, got {off5}"
expected_rem_split = [IntervalMap(10,2), IntervalMap(15,5)]
assert list_iv_equal(rem5, expected_rem_split), f"apply_rule inside: remainder split wrong, got {rem5}"

print("Test 6")
# 6) apply_multiple_rules: combine rules that cover different parts
rules = ["100 12 3", "0 15 10"]  # first extracts middle [12,14], second handles right [15,19]
transformed = self2.apply_multiple_rules(rules, False)
# Expect transformed pieces: offsetted middle [100,3], offsetted right [0,5], and leftover left [10,2]
expected_pieces = {(100,3), (0,5), (10,2)}

def to_tuple_set(iv_list):
    return {(iv.start, iv.length) for iv in iv_list}

assert to_tuple_set(transformed) == expected_pieces, f"apply_multiple_rules produced {to_tuple_set(transformed)}, expected {expected_pieces}"

print("All rule-based tests passed")

Test 1
Test 2
Test 3
Test 4
Test 5
Test 6
All rule-based tests passed


## Solving

In [63]:
seeds, data_dict = process_data(data_parsed)

def transform_seeds_to_intervalmaps(seeds):
    starts, lengths = seeds[::2], seeds[1::2]

    return [IntervalMap(start, length) for start, length in zip(starts, lengths)]

rules_order = {
        1: "seed-to-soil",
        2: "soil-to-fertilizer",
        3: "fertilizer-to-water",
        4: "water-to-light",
        5: "light-to-temperature",
        6: "temperature-to-humidity",
        7: "humidity-to-location",
    }

seeds_intervals = transform_seeds_to_intervalmaps(seeds)

In [64]:
result_2 = [seeds_intervals]
to_process = True
access_methods = IntervalMap(0, 1)
while len(result_2) < 8:
    idx = len(result_2)
    rules = data_dict[rules_order[idx]]
    to_process = result_2[-1]
    result_2.append(access_methods.multi_intervals_multi_rules(to_process, rules))


In [66]:
access_methods.get_min_of_intervals(result_2[-1])

136096660